# Ingredient co-occurrence graph

This notebook builds a weighted co-occurrence graph between ingredients based on
which ingredients appear together in the same recipe.

The output is an edge list that will be reused in later stages of the project.


In [2]:
# -- Setup --
# Import the libraries needed. All paths used in this notebook are defined here so they can be changed easily if needed.

from pathlib import Path
import pandas as pd
import ast

# Directory containing the processed outputs from the data cleaning stage of the project
OUT_DIR = Path("../data/processed")

# Cleaned recipes table:
# one row per recipe, with a list of cleaned ingredients
RECIPE_INGREDIENTS_PATH = OUT_DIR / "recipe_ingredients.csv"

# Output file for this notebook:
# weighted ingredient co-occurrence edge list
EDGES_PATH = OUT_DIR / "ingredient_cooccurrence_edges.csv"


# Load cleaned recipes


In [3]:
# Read the cleaned recipes table produced in the previous stage.
# The ingredient lists were written to CSV as strings, so they need to be converted back into Python lists before use.

# Load the table
recipes_df = pd.read_csv(RECIPE_INGREDIENTS_PATH)

# Basic information to confirm the file loaded as expected
print("Loaded recipe_ingredients.csv")
print("Number of recipes:", len(recipes_df))
print("Columns:", list(recipes_df.columns))

# Restore the ingredients column from string representation
# (e.g. "['egg', 'milk']") to an actual list object
recipes_df["ingredients"] = recipes_df["ingredients"].apply(ast.literal_eval)

# Quick visual check of the data
recipes_df.head()


Loaded recipe_ingredients.csv
Number of recipes: 219890
Columns: ['id', 'ingredients', 'ingredients_raw', 'n_clean_ingredients']


,id,ingredients,ingredients_raw,n_clean_ingredients
0,137739,"[honey, mexican, mixed spice, winter squash]","['winter squash', 'mexican seasoning', 'mixed ...",4
1,31490,"[cheese, egg, milk, pizza crust]","['prepared pizza crust', 'sausage patty', 'egg...",4
2,112140,"[beef, cheddar cheese, chili powder, cumin, ki...","['ground beef', 'yellow onions', 'diced tomato...",9
3,59389,"[herb, new potato, parsley, red pepper, red wi...","['spreadable cheese with garlic and herbs', 'n...",7
4,44061,"[apple cider vinegar, mustard, tomato juice]","['tomato juice', 'apple cider vinegar', 'sugar...",3


# What an edge represents (example/optional)


In [4]:
# -- Example: co-occurrence pairs from a single recipe -- 
# This cell is only for illustration. It shows how a single recipe is turned into a set of ingredient pairs ("edges")
# in the co-occurrence graph.

from itertools import combinations

# Select one recipe to inspect
row_idx = 0

# Ingredient list for this recipe
ingredients = recipes_df.loc[row_idx, "ingredients"]

print("Recipe id:", recipes_df.loc[row_idx, "id"])
print("Ingredients:", ingredients)

# Remove any duplicates within the recipe and sort the list
# so the pair representation is consistent
unique_ingredients = sorted(set(ingredients))
print("Unique ingredients:", unique_ingredients)

# Generate all unordered ingredient pairs
pairs = list(combinations(unique_ingredients, 2))

print("\nCo-occurrence pairs from this recipe:")
for a, b in pairs:
    print(f"({a}, {b})")

print("\nNumber of pairs:", len(pairs))


Recipe id: 137739
Ingredients: ['honey', 'mexican', 'mixed spice', 'winter squash']
Unique ingredients: ['honey', 'mexican', 'mixed spice', 'winter squash']

Co-occurrence pairs from this recipe:
(honey, mexican)
(honey, mixed spice)
(honey, winter squash)
(mexican, mixed spice)
(mexican, winter squash)
(mixed spice, winter squash)

Number of pairs: 6


# Co-occurrence counts on a small subset


In [5]:
# -- Co-occurrence counts on a small subset --
# This step runs the co-occurrence logic on a limited number of recipes. The goal is not performance, but to confirm that:
#   - pairs are counted correctly,
#   - each recipe contributes at most one count per pair,
#   - the most frequent pairs look reasonable.

from itertools import combinations
from collections import Counter

# Number of recipes used for this sanity check
SUBSET_N = 1000

# Mapping: (ingredient_a, ingredient_b) -> number of recipes
edge_counts = Counter()

# Process only a small subset of the data
for ingredients in recipes_df["ingredients"].head(SUBSET_N):
    # Ensure each ingredient is only counted once per recipe
    unique_ingredients = sorted(set(ingredients))

    # Generate all unordered pairs and update counts
    for a, b in combinations(unique_ingredients, 2):
        edge_counts[(a, b)] += 1

# Summary of the subset run
print("Subset size (recipes):", SUBSET_N)
print("Number of unique edges found:", len(edge_counts))

# Inspect the most frequent pairs as a quick sanity check
print("\nTop 20 edges in subset:")
for (a, b), count in edge_counts.most_common(20):
    print(f"({a}, {b}) -> {count}")


Subset size (recipes): 1000
Number of unique edges found: 13882

Top 20 edges in subset:
(egg, flour) -> 145
(flour, milk) -> 89
(flour, powder) -> 75
(flour, soda) -> 75
(egg, milk) -> 72
(flour, vanilla) -> 65
(egg, powder) -> 64
(egg, vanilla) -> 61
(egg, soda) -> 58
(cinnamon, flour) -> 46
(cinnamon, egg) -> 45
(garlic clove, tomato) -> 40
(powder, soda) -> 38
(soda, vanilla) -> 37
(milk, powder) -> 32
(milk, vanilla) -> 32
(powder, vanilla) -> 31
(cream, egg) -> 31
(cinnamon, soda) -> 30
(cream, flour) -> 29


# Full co-occurrence edge list


In [6]:
#  -- Full co-occurrence edge list --
# Build the final ingredient co-occurrence edge list using the full dataset. This is the version that will be saved to disk
# and reused by later stages of the project.
#
# Key points:
# - ingredient pairs are treated as unordered
# - each recipe contributes at most one count per pair
# - the resulting edge weights represent recipe-level co-occurrence

from itertools import combinations
from collections import Counter

# Stores: (ingredient_a, ingredient_b) -> number of recipes
edge_counts = Counter()

# Process all recipes
for ingredients in recipes_df["ingredients"]:
    # Remove any duplicates within a recipe and keep ordering stable
    unique_ingredients = sorted(set(ingredients))

    # Update counts for all unordered ingredient pairs
    for a, b in combinations(unique_ingredients, 2):
        edge_counts[(a, b)] += 1

print("Finished counting co-occurrence edges.")
print("Number of unique edges:", len(edge_counts))


# -- Convert counts to a table and save -- 
# The edge list is stored in a simple tabular format so it can
# be loaded easily by graph and embedding code later on.

edges_df = pd.DataFrame(
    [
        {"ingredient_a": a, "ingredient_b": b, "weight": count}
        for (a, b), count in edge_counts.items()
    ]
)

# Sort by weight so the strongest co-occurrences appear first
edges_df = edges_df.sort_values(by="weight", ascending=False).reset_index(drop=True)

# Quick visual check
edges_df.head()


# Write the edge list to disk
edges_df.to_csv(EDGES_PATH, index=False)

print(f"Saved co-occurrence edges to: {EDGES_PATH}")


Finished counting co-occurrence edges.
Number of unique edges: 419913
Saved co-occurrence edges to: ../data/processed/ingredient_cooccurrence_edges.csv


In [7]:
# -- Quick sanity checks on the saved edge list -- 
# Purpose:
# Make sure the file wrote correctly and the edge list has the properties we rely on before moving on (graph build + Node2Vec).

edges_check = pd.read_csv(EDGES_PATH)

print("Loaded:", EDGES_PATH)
print("Rows (edges):", len(edges_check))
print("Columns:", list(edges_check.columns))
display(edges_check.head(10))

# Basic schema checks
assert set(edges_check.columns) == {"ingredient_a", "ingredient_b", "weight"}, "Unexpected columns."

# No self-loops (an ingredient paired with itself)
self_loops = (edges_check["ingredient_a"] == edges_check["ingredient_b"]).sum()
print("Self-loops:", self_loops)
assert self_loops == 0, "Found self-loops."

# Weights should be positive integers
bad_weights = ((edges_check["weight"] <= 0) | (edges_check["weight"] % 1 != 0)).sum()
print("Bad weights:", bad_weights)
assert bad_weights == 0, "Found non-positive or non-integer weights."

# Spot-check distribution (helps catch obviously broken counts)
print("\nWeight summary:")
print(edges_check["weight"].describe())

print("\nTop 10 strongest edges:")
display(edges_check.sort_values("weight", ascending=False).head(10))

print("\nEdge list looks consistent.")


Loaded: ../data/processed/ingredient_cooccurrence_edges.csv
Rows (edges): 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,egg,flour,27623
1,egg,milk,16380
2,flour,milk,16133
3,flour,powder,15622
4,egg,powder,13475
5,flour,soda,12537
6,egg,soda,11069
7,cinnamon,flour,9311
8,cinnamon,egg,9240
9,egg,vanilla,8812


Self-loops: 0
Bad weights: 0

Weight summary:
count    419913.000000
mean         12.905892
std         114.985514
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max       27623.000000
Name: weight, dtype: float64

Top 10 strongest edges:


,ingredient_a,ingredient_b,weight
0,egg,flour,27623
1,egg,milk,16380
2,flour,milk,16133
3,flour,powder,15622
4,egg,powder,13475
5,flour,soda,12537
6,egg,soda,11069
7,cinnamon,flour,9311
8,cinnamon,egg,9240
9,egg,vanilla,8812



Edge list looks consistent.


In [9]:
# -----------------------------------------------------------
# H.0 — Compute ingredient marginal frequencies (df) for weighting
# (Robust to pandas naming differences)
# -----------------------------------------------------------

DF_PATH = OUT_DIR / "ingredient_df.csv"

# 1) Convert ingredient lists to sets (one vote per ingredient per recipe)
ingredient_sets = recipes_df["ingredients"].apply(lambda xs: set(xs))

# 2) Explode so we get one row per ingredient occurrence (per recipe)
ingredient_long = ingredient_sets.explode()

# 3) Count how many recipes each ingredient appears in
vc = ingredient_long.value_counts()

print("value_counts() produced a Series named:", vc.name)
print("value_counts() head:")
print(vc.head())

ingredient_df = vc.reset_index()

# Debug: see what pandas called the columns
print("\nAfter reset_index(), columns are:", list(ingredient_df.columns))
display(ingredient_df.head(5))

# 4) Rename columns to a stable schema: ingredient, df
# In many pandas versions the columns become: ["index", "count"] OR ["ingredient", 0]
col0, col1 = ingredient_df.columns[0], ingredient_df.columns[1]
ingredient_df = ingredient_df.rename(columns={col0: "ingredient", col1: "df"})

# Debug: confirm schema
print("\nAfter rename, columns are:", list(ingredient_df.columns))
display(ingredient_df.head(5))

# 5) Sort by df (most frequent ingredients first)
ingredient_df = ingredient_df.sort_values("df", ascending=False).reset_index(drop=True)

# 6) Save to disk
ingredient_df.to_csv(DF_PATH, index=False)

print("Saved ingredient document frequencies to:", DF_PATH)
print("Number of unique ingredients:", len(ingredient_df))

print("\nTop 20 most frequent ingredients (by #recipes):")
display(ingredient_df.head(20))

# 7) Basic sanity checks (match previous notebook style)
assert set(ingredient_df.columns) == {"ingredient", "df"}, "Unexpected columns."
assert (ingredient_df["df"] > 0).all(), "Found non-positive df values."


value_counts() produced a Series named: count
value_counts() head:
ingredients
egg             51352
flour           49386
garlic clove    36422
milk            35664
tomato          23071
Name: count, dtype: int64

After reset_index(), columns are: ['ingredients', 'count']


,ingredients,count
0,egg,51352
1,flour,49386
2,garlic clove,36422
3,milk,35664
4,tomato,23071



After rename, columns are: ['ingredient', 'df']


,ingredient,df
0,egg,51352
1,flour,49386
2,garlic clove,36422
3,milk,35664
4,tomato,23071


Saved ingredient document frequencies to: ../data/processed/ingredient_df.csv
Number of unique ingredients: 2698

Top 20 most frequent ingredients (by #recipes):


,ingredient,df
0,egg,51352
1,flour,49386
2,garlic clove,36422
3,milk,35664
4,tomato,23071
5,chicken,22644
6,cream,21906
7,garlic,19606
8,lemon juice,19573
9,cinnamon,19137


In [10]:
# -----------------------------------------------------------
# H.1 — Build a NORMALISED co-occurrence edge list (Association Strength)
# -----------------------------------------------------------
# Goal:
# Convert raw co-occurrence counts c_ij into a weight that down-weights "hub" ingredients
# (very common items like egg, salt, flour) so the graph reflects *association* rather
# than just popularity.
#
# We use "association strength" (AS) style normalisation:
#   w_ij = c_ij / (df_i * df_j)
#
# where:
# - c_ij = number of recipes containing both i and j (your raw co-occurrence count)
# - df_i = number of recipes containing i (ingredient document frequency)
#
# This is widely used in co-occurrence normalisation (bibliometrics / co-word analysis)
# and is exactly the kind of "scientifically supported" move you can justify in a dissertation.

# Path for the NEW normalised edge list (do NOT overwrite the raw one)
NORM_EDGES_PATH = OUT_DIR / "ingredient_cooccurrence_edges_assoc_strength.csv"

# 1) Create a fast lookup: ingredient -> df
# (Using a dict makes the next step efficient and easy to read.)
df_lookup = dict(zip(ingredient_df["ingredient"], ingredient_df["df"]))

# Quick sanity check (optional)
print("df_lookup size:", len(df_lookup))
print("Example df(egg):", df_lookup.get("egg"))

# 2) Convert the raw edge counter into a DataFrame
# (This preserves your existing co-occurrence logic; we are only changing the weights.)
edges_norm_df = pd.DataFrame(
    [{"ingredient_a": a, "ingredient_b": b, "count": c}
     for (a, b), c in edge_counts.items()]
)

print("Raw edges table built.")
print("Rows (edges):", len(edges_norm_df))
display(edges_norm_df.head(5))

# 3) Attach df for each endpoint of the edge
# If an ingredient is missing from df_lookup (shouldn't happen), we'll get NaN.
edges_norm_df["df_a"] = edges_norm_df["ingredient_a"].map(df_lookup)
edges_norm_df["df_b"] = edges_norm_df["ingredient_b"].map(df_lookup)

# Sanity: check for missing df values
missing_df = edges_norm_df["df_a"].isna().sum() + edges_norm_df["df_b"].isna().sum()
print("Missing df values (should be 0):", missing_df)

# 4) Compute the association-strength weight
# Using float weights is fine for NetworkX + node2vec (weight_key expects numeric).
edges_norm_df["weight"] = edges_norm_df["count"] / (edges_norm_df["df_a"] * edges_norm_df["df_b"])

# 5) Clean + sort
# - Drop any rows where df is missing (defensive)
# - Sort by new weight so the strongest *associations* appear first
edges_norm_df = edges_norm_df.dropna(subset=["df_a", "df_b", "weight"])
edges_norm_df = edges_norm_df.sort_values("weight", ascending=False).reset_index(drop=True)

# Quick check: strongest associations (these should look less "hubby" than raw counts)
print("\nTop 10 normalised edges (association strength):")
display(edges_norm_df[["ingredient_a", "ingredient_b", "count", "df_a", "df_b", "weight"]].head(10))

# 6) Save the normalised edge list (only the columns later steps need)
edges_norm_df[["ingredient_a", "ingredient_b", "weight"]].to_csv(NORM_EDGES_PATH, index=False)

print("\nSaved normalised edge list to:", NORM_EDGES_PATH)
print("Rows saved:", len(edges_norm_df))

# 7) Safety checks (match your style)
edges_norm_check = pd.read_csv(NORM_EDGES_PATH)
assert set(edges_norm_check.columns) == {"ingredient_a", "ingredient_b", "weight"}, "Unexpected columns."
assert (edges_norm_check["weight"] > 0).all(), "Found non-positive weights."
print("Normalised edge list looks consistent.")


df_lookup size: 2698
Example df(egg): 51352
Raw edges table built.
Rows (edges): 419913


,ingredient_a,ingredient_b,count
0,honey,mexican,2
1,honey,mixed spice,16
2,honey,winter squash,3
3,mexican,mixed spice,1
4,mexican,winter squash,1


Missing df values (should be 0): 0

Top 10 normalised edges (association strength):


,ingredient_a,ingredient_b,count,df_a,df_b,weight
0,vegan cream cheese,vegan parmesan cheese,5,24,20,0.010417
1,corn husk,masa harina flour,12,34,36,0.009804
2,capicola,mortadella,8,26,35,0.008791
3,corn chex,wheat chex,37,90,50,0.008222
4,rice chex,wheat chex,40,107,50,0.007477
5,channa dal,mung dal,5,35,20,0.007143
6,butter pecan cake,coconut pecan frosting,3,21,21,0.006803
7,paper cup,wooden popsicle stick,12,38,48,0.006579
8,nori,sushi rice,37,103,55,0.006531
9,hing,mung dal,3,23,20,0.006522



Saved normalised edge list to: ../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv
Rows saved: 419913
Normalised edge list looks consistent.


In [12]:
# -----------------------------------------------------------
# H.2 — Build a "best-of-both-worlds" edge list:
# prune by association strength, but use log(count) as Node2Vec walk weights
# -----------------------------------------------------------
# Motivation:
# - Assoc-strength weights can become extremely tiny for common pairs (egg–flour etc.),
#   which makes weighted Node2Vec almost ignore them.
# - Instead, we use assoc-strength ONLY to decide which edges are worth keeping (pruning),
#   then we use log1p(count) as the actual walk weight for Node2Vec.

import numpy as np
import pandas as pd

# Input: association-strength edge list built above (read-only)
NORM_EDGES_PATH = OUT_DIR / "ingredient_cooccurrence_edges_assoc_strength.csv"

# Output: pruned graph that Node2Vec will actually use
PRUNED_EDGES_PATH = OUT_DIR / "ingredient_edges_pruned_by_assoc_top50_logcount.csv"

# 1) Load association-strength edges
assoc_df = pd.read_csv(NORM_EDGES_PATH)

print("Loaded assoc-strength edges:", NORM_EDGES_PATH)
print("Rows:", len(assoc_df))
print("Columns:", list(assoc_df.columns))
display(assoc_df.head(5))

# 2) Rebuild counts alongside assoc weights (needed for log1p(count))
# We still have edge_counts in memory from earlier cells, so we reconstruct a count table.
counts_df = pd.DataFrame(
    [{"ingredient_a": a, "ingredient_b": b, "count": c} for (a, b), c in edge_counts.items()]
)

# Merge: (ingredient_a, ingredient_b) -> [weight (assoc), count]
assoc_df = assoc_df.merge(counts_df, on=["ingredient_a", "ingredient_b"], how="left")

missing_counts = assoc_df["count"].isna().sum()
print("Missing counts after merge (should be 0):", missing_counts)
assert missing_counts == 0, "Some assoc edges did not match raw count edges."

# 3) Keep TOP-K neighbours per node by assoc strength
TOPK = 50

# Create a directed view so we can take top-K per node, then re-undirect it.
a_view = assoc_df[["ingredient_a", "ingredient_b", "count", "weight"]].rename(
    columns={"ingredient_a": "src", "ingredient_b": "dst"}
)
b_view = assoc_df[["ingredient_b", "ingredient_a", "count", "weight"]].rename(
    columns={"ingredient_b": "src", "ingredient_a": "dst"}
)
directed = pd.concat([a_view, b_view], ignore_index=True)

directed_topk = (
    directed.sort_values(["src", "weight"], ascending=[True, False])
            .groupby("src", as_index=False)
            .head(TOPK)
)

print("Directed edges after top-K pruning:", len(directed_topk))

# 4) Convert back to undirected edges by sorting endpoints
u = np.minimum(directed_topk["src"].values, directed_topk["dst"].values)
v = np.maximum(directed_topk["src"].values, directed_topk["dst"].values)

undirected_topk = directed_topk.copy()
undirected_topk["ingredient_a"] = u
undirected_topk["ingredient_b"] = v

# Keep an edge if it survives from either endpoint’s top-K list
# If duplicates happen, keep the maximum assoc weight and maximum count.
pruned = (
    undirected_topk.groupby(["ingredient_a", "ingredient_b"], as_index=False)
                  .agg({"weight": "max", "count": "max"})
)

print("Undirected edges after pruning:", len(pruned))
display(pruned.sort_values("weight", ascending=False).head(10))

# 5) Define Node2Vec walk weights as log(1 + count)
pruned["walk_weight"] = np.log1p(pruned["count"].astype(float))

# 6) Save in the same schema your Node2Vec code expects: ingredient_a, ingredient_b, weight
out_df = pruned[["ingredient_a", "ingredient_b", "walk_weight"]].rename(columns={"walk_weight": "weight"})
out_df.to_csv(PRUNED_EDGES_PATH, index=False)

print("Saved pruned edge list to:", PRUNED_EDGES_PATH)
print("Rows saved:", len(out_df))
display(out_df.sort_values("weight", ascending=False).head(10))

# 7) Sanity checks (match your style)
check = pd.read_csv(PRUNED_EDGES_PATH)
assert set(check.columns) == {"ingredient_a", "ingredient_b", "weight"}, "Unexpected columns."
assert (check["ingredient_a"] != check["ingredient_b"]).all(), "Found self-loops."
assert (check["weight"] > 0).all(), "Found non-positive weights."

print("Pruned log-count edge list looks consistent.")


Loaded assoc-strength edges: ../data/processed/ingredient_cooccurrence_edges_assoc_strength.csv
Rows: 419913
Columns: ['ingredient_a', 'ingredient_b', 'weight']


,ingredient_a,ingredient_b,weight
0,vegan cream cheese,vegan parmesan cheese,0.010417
1,corn husk,masa harina flour,0.009804
2,capicola,mortadella,0.008791
3,corn chex,wheat chex,0.008222
4,rice chex,wheat chex,0.007477


Missing counts after merge (should be 0): 0
Directed edges after top-K pruning: 134648
Undirected edges after pruning: 81362


,ingredient_a,ingredient_b,weight,count
81238,vegan cream cheese,vegan parmesan cheese,0.010417,5
36301,corn husk,masa harina flour,0.009804,12
22207,capicola,mortadella,0.008791,8
36203,corn chex,wheat chex,0.008222,37
76464,rice chex,wheat chex,0.007477,40
25186,channa dal,mung dal,0.007143,5
19035,butter pecan cake,coconut pecan frosting,0.006803,3
69906,paper cup,wooden popsicle stick,0.006579,12
67843,nori,sushi rice,0.006531,37
55524,hing,mung dal,0.006522,3


Saved pruned edge list to: ../data/processed/ingredient_edges_pruned_by_assoc_top50_logcount.csv
Rows saved: 81362


,ingredient_a,ingredient_b,weight
47612,flour,powder,9.656499
43828,egg,powder,9.508666
47623,flour,soda,9.436519
43839,egg,soda,9.311994
74150,powder,soda,8.818038
32698,cinnamon,nutmeg,8.513787
79192,soda,vanilla,8.398184
23252,carrot,celery,8.222016
9674,basil,oregano,8.143517
29175,chili powder,cumin,8.080856


Pruned log-count edge list looks consistent.


In [13]:
# sanity: log1p(count) should match the saved weight
test_pair = ("flour", "powder")

# get the count from your original edges_check (or from edge_counts dict)
c = edge_counts[tuple(sorted(test_pair))]
print("count:", c)
print("log1p(count):", np.log1p(c))

# find the saved edge weight
saved = pd.read_csv(PRUNED_EDGES_PATH)
row = saved[(saved["ingredient_a"]=="flour") & (saved["ingredient_b"]=="powder")]
print(row)


count: 15622
log1p(count): 9.656499466411903
      ingredient_a ingredient_b    weight
47612        flour       powder  9.656499


In [14]:
# ===========================================================
# Ingredient–Recipe Bipartite Graph (LOSSLESS, UNWEIGHTED)
# ===========================================================
# Goal (per our research decision):
# - Build a bipartite graph with TWO node types:
#     1) ingredient nodes (strings like "milk", "garlic")
#     2) recipe nodes (one per recipe)
# - Add an edge (ingredient) -- (recipe) iff ingredient appears in recipe
# - IMPORTANT DESIGN CHOICES (matching the research):
#     - NO projection to ingredient–ingredient edges (avoids clique explosion)
#     - NO pruning, NO association strength, NO log weights (keeps connectivity)
#     - Unweighted edges (weight=1) so Node2Vec random walks naturally:
#         * downweight very large recipes (dilution through high-degree recipe nodes)
#         * diffuse hub ingredients (many recipe neighbors)
#
# Output:
# - A clean edge list you can feed to NetworkX / node2vec:
#     columns: node_u, node_v, weight
# - A nodes table with node types (ingredient vs recipe) for interpretability

from pathlib import Path
import pandas as pd
import ast

# ----------------------------
# Paths
# ----------------------------
OUT_DIR = Path("../data/processed")
RECIPE_INGREDIENTS_PATH = OUT_DIR / "recipe_ingredients.csv"

BIPARTITE_EDGES_PATH = OUT_DIR / "ingredient_recipe_bipartite_edges.csv"
BIPARTITE_NODES_PATH = OUT_DIR / "ingredient_recipe_bipartite_nodes.csv"

# ----------------------------
# Load cleaned recipes
# ----------------------------
recipes_df = pd.read_csv(RECIPE_INGREDIENTS_PATH)

print("Loaded recipe_ingredients.csv")
print("Number of recipes:", len(recipes_df))
print("Columns:", list(recipes_df.columns))

# Restore list-of-ingredients column (stored as string in CSV)
recipes_df["ingredients"] = recipes_df["ingredients"].apply(ast.literal_eval)

# Defensive: ensure we have a stable recipe identifier column
# Your file likely has "id". If not, we fallback to the dataframe index.
if "id" not in recipes_df.columns:
    print("WARNING: No 'id' column found. Using row index as recipe id.")
    recipes_df = recipes_df.reset_index().rename(columns={"index": "id"})

# ----------------------------
# Build bipartite edge list
# ----------------------------
# We create a dedicated namespace for recipe nodes to avoid collisions with ingredient strings.
# Example recipe node id: "recipe::123456"
def recipe_node_id(rid: int | str) -> str:
    return f"recipe::{rid}"

edge_rows = []

# For each recipe, add edges (recipe_node) -- (ingredient)
# Note: each ingredient counted at most once per recipe (deduplicate within recipe)
for rid, ing_list in zip(recipes_df["id"], recipes_df["ingredients"]):
    rnode = recipe_node_id(rid)
    unique_ings = sorted(set(ing_list))
    for ing in unique_ings:
        edge_rows.append({"node_u": rnode, "node_v": ing, "weight": 1})

edges_df = pd.DataFrame(edge_rows)

print("\nBuilt bipartite edges.")
print("Rows (edges):", len(edges_df))
print("Columns:", list(edges_df.columns))
display(edges_df.head(10))

# ----------------------------
# Save edge list
# ----------------------------
edges_df.to_csv(BIPARTITE_EDGES_PATH, index=False)
print(f"\nSaved bipartite edge list to: {BIPARTITE_EDGES_PATH}")

# ----------------------------
# Build nodes table (for interpretability / sanity)
# ----------------------------
# Node types:
# - recipe nodes start with "recipe::"
# - ingredient nodes are everything else
recipe_nodes = edges_df["node_u"].unique()   # node_u is always recipe node in our construction
ingredient_nodes = edges_df["node_v"].unique()

nodes_df = pd.DataFrame(
    [{"node": n, "node_type": "recipe"} for n in recipe_nodes] +
    [{"node": n, "node_type": "ingredient"} for n in ingredient_nodes]
)

nodes_df.to_csv(BIPARTITE_NODES_PATH, index=False)
print(f"Saved bipartite nodes table to: {BIPARTITE_NODES_PATH}")

# ----------------------------
# Sanity checks (match your style)
# ----------------------------
edges_check = pd.read_csv(BIPARTITE_EDGES_PATH)

assert set(edges_check.columns) == {"node_u", "node_v", "weight"}, "Unexpected columns."

# No self-loops
self_loops = (edges_check["node_u"] == edges_check["node_v"]).sum()
print("\nSelf-loops:", self_loops)
assert self_loops == 0, "Found self-loops."

# Weights are positive integers (=1)
bad_weights = ((edges_check["weight"] <= 0) | (edges_check["weight"] % 1 != 0)).sum()
print("Bad weights:", bad_weights)
assert bad_weights == 0, "Found non-positive or non-integer weights."

# Check that node_u are recipe nodes and node_v are ingredients (by construction)
bad_u = (~edges_check["node_u"].astype(str).str.startswith("recipe::")).sum()
bad_v = (edges_check["node_v"].astype(str).str.startswith("recipe::")).sum()
print("Edges with non-recipe node_u (should be 0):", bad_u)
print("Edges with recipe-like node_v (should be 0):", bad_v)
assert bad_u == 0, "Found non-recipe ids in node_u."
assert bad_v == 0, "Found recipe ids in node_v."

# Basic stats
num_recipe_nodes = edges_check["node_u"].nunique()
num_ingredient_nodes = edges_check["node_v"].nunique()
num_edges = len(edges_check)

print("\nBipartite graph stats:")
print("Recipe nodes:", num_recipe_nodes)
print("Ingredient nodes:", num_ingredient_nodes)
print("Total nodes:", num_recipe_nodes + num_ingredient_nodes)
print("Edges:", num_edges)

# Degree summaries (useful to confirm the intended 'natural downweighting' behavior)
# - recipe degree = number of unique ingredients in recipe
# - ingredient degree = number of recipes containing ingredient
recipe_deg = edges_check["node_u"].value_counts()
ingredient_deg = edges_check["node_v"].value_counts()

print("\nRecipe degree summary (ingredients per recipe):")
print(recipe_deg.describe())

print("\nIngredient degree summary (recipes per ingredient):")
print(ingredient_deg.describe())

print("\nEdge list looks consistent and matches the bipartite design.")


Loaded recipe_ingredients.csv
Number of recipes: 219890
Columns: ['id', 'ingredients', 'ingredients_raw', 'n_clean_ingredients']

Built bipartite edges.
Rows (edges): 1522980
Columns: ['node_u', 'node_v', 'weight']


,node_u,node_v,weight
0,recipe::137739,honey,1
1,recipe::137739,mexican,1
2,recipe::137739,mixed spice,1
3,recipe::137739,winter squash,1
4,recipe::31490,cheese,1
5,recipe::31490,egg,1
6,recipe::31490,milk,1
7,recipe::31490,pizza crust,1
8,recipe::112140,beef,1
9,recipe::112140,cheddar cheese,1



Saved bipartite edge list to: ../data/processed/ingredient_recipe_bipartite_edges.csv
Saved bipartite nodes table to: ../data/processed/ingredient_recipe_bipartite_nodes.csv

Self-loops: 0
Bad weights: 0
Edges with non-recipe node_u (should be 0): 0
Edges with recipe-like node_v (should be 0): 0

Bipartite graph stats:
Recipe nodes: 219890
Ingredient nodes: 2698
Total nodes: 222588
Edges: 1522980

Recipe degree summary (ingredients per recipe):
count    219890.000000
mean          6.926099
std           2.871718
min           3.000000
25%           5.000000
50%           6.000000
75%           9.000000
max          39.000000
Name: count, dtype: float64

Ingredient degree summary (recipes per ingredient):
count     2698.000000
mean       564.484804
std       2410.261556
min         10.000000
25%         35.000000
50%         76.000000
75%        237.000000
max      51352.000000
Name: count, dtype: float64

Edge list looks consistent and matches the bipartite design.
